# M3U Playlist Downloader

A utility for downloading M3U playlists with support for various IPTV provider formats including Xtream codes. This tool is designed to work with OTT Navigator, NS Player, and other IPTV applications.

## Features

- Downloads M3U playlists from direct URLs
- Supports Xtream codes API format
- Handles authentication via username/password
- Multiple user-agents for compatibility with various providers
- Fixes common formatting issues in playlists
- Fallback methods for difficult providers

## Requirements

- Python 3.6+
- requests library
- urllib library


## Import Libraries

In [ ]:
import requests
import time
import re
import os
import urllib.parse

# For GitHub compatibility, we need to handle the Google Colab specific imports
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False
    # Define a simple placeholder for files.download when not in Colab
    class MockFiles:
        @staticmethod
        def download(filename):
            print(f"File saved at: {filename}")
            print("Note: Running outside of Colab, so file can't be automatically downloaded.")
            print("Please manually download the file from your working directory.")
    
    files = MockFiles()

## Main Playlist Download Function

This function handles the downloading of M3U playlists with support for various provider formats.

In [ ]:
def download_m3u_playlist(url, username=None, password=None, output_file="playlist.m3u"):
    """
    Download M3U playlist with support for various IPTV provider formats
    
    Args:
        url (str): The playlist URL or Xtream codes API URL
        username (str, optional): Username for authentication
        password (str, optional): Password for authentication
        output_file (str): Name of output file
    """
    
    # Set of User-Agents to try if initial download fails
    user_agents = [
        # OTT Navigator specific user agent
        "Dalvik/2.1.0 (Linux; U; Android 9; OTT-Navigator)",
        # NS Player specific user agent
        "NSPlayer/12.00.14393.3542",
        # Other common IPTV player agents
        "VLC/3.0.16 LibVLC/3.0.16",
        "Mozilla/5.0 (QtEmbedded; U; Linux; C) AppleWebKit/533.3 (KHTML, like Gecko) MAG200 stbapp ver: 4 rev: 2721 Mobile Safari/533.3",
        "IPTV-Smarters",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
    ]
    
    # Check if URL contains username and password parameters
    parsed_url = urllib.parse.urlparse(url)
    query_params = urllib.parse.parse_qs(parsed_url.query)
    
    # If username/password are provided in function but not in URL
    if username and password and 'username' not in query_params and 'password' not in query_params:
        # Check if URL follows Xtream codes format
        if '/get.php' in url or parsed_url.path.endswith('.php'):
            separator = '&' if '?' in url else '?'
            url = f"{url}{separator}username={username}&password={password}"
            
            # Add Xtream codes specific parameters if not present
            if 'type' not in query_params:
                url += "&type=m3u_plus"
            if 'output' not in query_params:
                url += "&output=ts"
    
    print(f"Attempting to download playlist from: {url}")
    
    # Try with different user agents
    for user_agent in user_agents:
        try:
            print(f"\nTrying with User-Agent: {user_agent}")
            
            headers = {
                'User-Agent': user_agent,
                'Accept': '*/*',
                'Connection': 'keep-alive'
            }
            
            # Add specific headers needed for some providers
            if "OTT-Navigator" in user_agent:
                headers['Accept-Encoding'] = 'gzip, deflate'
                headers['X-Playback-Session-Id'] = 'ott-navigator-session'
            
            response = requests.get(url, headers=headers, timeout=30)
            
            if response.status_code == 200:
                content = response.text
                
                # Verify it's actually an M3U file
                if content.strip().startswith('#EXTM3U'):
                    print("Successfully downloaded M3U playlist!")
                    
                    # Save the playlist file
                    with open(output_file, 'w', encoding='utf-8') as f:
                        f.write(content)
                    
                    print(f"Playlist saved to {output_file}")
                    
                    # Fix common formatting issues for better compatibility
                    fix_playlist_for_compatibility(output_file)
                    
                    # Download to local machine
                    files.download(output_file)
                    return True
                else:
                    print("Response doesn't appear to be a valid M3U playlist.")
                    print(f"Content starts with: {content[:100].strip()}")
            else:
                print(f"Failed with status code: {response.status_code}")
                if response.status_code == 404:
                    print("Error 404: Playlist URL not found. Check if the URL is correct and active.")
                elif response.status_code == 401 or response.status_code == 403:
                    print("Authentication error. Check your username and password.")
                elif response.status_code == 503:
                    print("Service unavailable. The server might be blocking your request.")
                    
        except Exception as e:
            print(f"Error: {str(e)}")
    
    print("\nAll attempts failed. Trying alternative methods...")
    
    # Try alternative download method using Xtream codes API format
    if try_xtream_codes_format(url, username, password, output_file):
        return True
    
    # Try curl as last resort
    try_curl_download(url, output_file)
    return False

## Helper Functions

The following functions assist in fixing playlist formatting and providing alternative download methods.

In [ ]:
def fix_playlist_for_compatibility(filename):
    """Fix common formatting issues in M3U playlists for better compatibility"""
    
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Make sure file starts with #EXTM3U
        if not content.strip().startswith('#EXTM3U'):
            content = '#EXTM3U\n' + content
        
        # Fix common issues with channel entries
        lines = content.split('\n')
        fixed_lines = []
        i = 0
        
        while i < len(lines):
            line = lines[i].strip()
            
            # Skip empty lines
            if not line:
                i += 1
                continue
            
            # Fix EXTINF lines
            if line.startswith('#EXTINF:'):
                # Make sure EXTINF has duration parameter
                if ':' in line and not re.match(r'#EXTINF:-?\d+', line):
                    line = re.sub(r'#EXTINF:', '#EXTINF:-1', line)
                
                # Make sure there's a URL on the next line
                next_line_index = i + 1
                while next_line_index < len(lines) and (not lines[next_line_index].strip() or lines[next_line_index].strip().startswith('#')):
                    next_line_index += 1
                
                if next_line_index < len(lines) and not lines[next_line_index].strip().startswith('http'):
                    # Skip this entry as it's invalid
                    i = next_line_index + 1
                    continue
            
            fixed_lines.append(line)
            i += 1
        
        # Write fixed content back to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(fixed_lines))
            
        print("Fixed playlist formatting for better compatibility")
        
    except Exception as e:
        print(f"Error fixing playlist: {str(e)}")

In [ ]:
def try_xtream_codes_format(url, username, password, output_file):
    """Try to download using Xtream codes API format"""
    
    if not username or not password:
        return False
    
    try:
        # Extract base URL
        parsed_url = urllib.parse.urlparse(url)
        base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
        
        # Try Xtream codes format
        xtream_url = f"{base_url}/get.php?username={username}&password={password}&type=m3u_plus&output=ts"
        
        print(f"\nTrying Xtream codes format: {xtream_url}")
        
        headers = {
            'User-Agent': 'OTT-Navigator/Dalvik',
            'Accept': '*/*'
        }
        
        response = requests.get(xtream_url, headers=headers, timeout=30)
        
        if response.status_code == 200 and response.text.strip().startswith('#EXTM3U'):
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            print(f"Successfully downloaded using Xtream codes format to {output_file}")
            files.download(output_file)
            return True
            
        return False
        
    except Exception as e:
        print(f"Error with Xtream codes format: {str(e)}")
        return False

In [ ]:
def try_curl_download(url, output_file):
    """Try to download using curl command as last resort"""
    
    try:
        print("\nTrying download with curl command...")
        # Check if we're in Colab (where we can use shell commands) or not
        if is_colab:
            # Use curl with OTT Navigator user agent
            !curl -A "Dalvik/2.1.0 (Linux; U; Android 9; OTT-Navigator)" "{url}" -o {output_file}
        else:
            # If not in Colab, use the requests library instead
            print("Not running in Colab, using requests library instead of curl")
            headers = {
                'User-Agent': 'Dalvik/2.1.0 (Linux; U; Android 9; OTT-Navigator)',
                'Accept': '*/*'
            }
            response = requests.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                with open(output_file, 'wb') as f:
                    f.write(response.content)
        
        # Check if file exists and has content
        if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
            with open(output_file, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read(100)
                if content.strip().startswith('#EXTM3U'):
                    print(f"Successfully downloaded with curl to {output_file}")
                    files.download(output_file)
                    return True
        
        print("Curl download failed or produced invalid M3U file")
        return False
        
    except Exception as e:
        print(f"Error with curl download: {str(e)}")
        return False

## Main Execution Function

This function handles user input and initiates the download process.

In [ ]:
def main():
    print("M3U Playlist Downloader for OTT Navigator and NS Player")
    print("-----------------------------------------------------")
    
    # Get user input
    url = input("Enter the M3U playlist URL: ")
    username = input("Enter username (if required, otherwise leave blank): ")
    password = input("Enter password (if required, otherwise leave blank): ")
    
    if not username:
        username = None
    if not password:
        password = None
    
    # Start download process
    download_m3u_playlist(url, username, password)

## Usage Example

Run the cell below to start the interactive M3U playlist downloader.

In [ ]:
# Run the main function
if __name__ == "__main__":
    main()